In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import plotly.graph_objs as go
from plotly.offline import init_notebook_mode, iplot
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from category_encoders import TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

In [2]:
import xgboost
print(xgboost.__version__)

3.2.0


In [3]:
app_train = pd.read_csv("C:/Users/myself/Projects/machine_learning/machine_learning_1/data/application_train.csv")
app_test = pd.read_csv("C:/Users/myself/Projects/machine_learning/machine_learning_1/data/application_test.csv")
prev_app = pd.read_csv("C:/Users/myself/Projects/machine_learning/machine_learning_1/data/previous_application.csv")
installments = pd.read_csv("C:/Users/myself/Projects/machine_learning/machine_learning_1/data/installments_payments.csv")
pos = pd.read_csv("C:/Users/myself/Projects/machine_learning/machine_learning_1/data/POS_CASH_balance.csv")
bureau = pd.read_csv("C:/Users/myself/Projects/machine_learning/machine_learning_1/data/bureau.csv")
bureau_bal = pd.read_csv("C:/Users/myself/Projects/machine_learning/machine_learning_1/data/bureau_balance.csv")
credit_card = pd.read_csv("C:/Users/myself/Projects/machine_learning/machine_learning_1/data/credit_card_balance.csv")

In [4]:
def info(df):  
    missing = (df.isnull().sum() / len(df)) * 100 
    missing = missing[missing > 0].sort_values(ascending=False)
    for col, perc in missing.items():
        print(f'Column "{col}" has {perc:.2f}% missing')    
    print(f'\n"{len(missing)}" columns have missing values')
        
info(app_train)

Column "COMMONAREA_MEDI" has 69.87% missing
Column "COMMONAREA_MODE" has 69.87% missing
Column "COMMONAREA_AVG" has 69.87% missing
Column "NONLIVINGAPARTMENTS_MODE" has 69.43% missing
Column "NONLIVINGAPARTMENTS_MEDI" has 69.43% missing
Column "NONLIVINGAPARTMENTS_AVG" has 69.43% missing
Column "FONDKAPREMONT_MODE" has 68.39% missing
Column "LIVINGAPARTMENTS_AVG" has 68.35% missing
Column "LIVINGAPARTMENTS_MEDI" has 68.35% missing
Column "LIVINGAPARTMENTS_MODE" has 68.35% missing
Column "FLOORSMIN_MEDI" has 67.85% missing
Column "FLOORSMIN_MODE" has 67.85% missing
Column "FLOORSMIN_AVG" has 67.85% missing
Column "YEARS_BUILD_MODE" has 66.50% missing
Column "YEARS_BUILD_MEDI" has 66.50% missing
Column "YEARS_BUILD_AVG" has 66.50% missing
Column "OWN_CAR_AGE" has 65.99% missing
Column "LANDAREA_AVG" has 59.38% missing
Column "LANDAREA_MEDI" has 59.38% missing
Column "LANDAREA_MODE" has 59.38% missing
Column "BASEMENTAREA_MODE" has 58.52% missing
Column "BASEMENTAREA_MEDI" has 58.52% miss

In [5]:
info(app_test)

Column "COMMONAREA_AVG" has 68.72% missing
Column "COMMONAREA_MODE" has 68.72% missing
Column "COMMONAREA_MEDI" has 68.72% missing
Column "NONLIVINGAPARTMENTS_AVG" has 68.41% missing
Column "NONLIVINGAPARTMENTS_MODE" has 68.41% missing
Column "NONLIVINGAPARTMENTS_MEDI" has 68.41% missing
Column "FONDKAPREMONT_MODE" has 67.28% missing
Column "LIVINGAPARTMENTS_MODE" has 67.25% missing
Column "LIVINGAPARTMENTS_AVG" has 67.25% missing
Column "LIVINGAPARTMENTS_MEDI" has 67.25% missing
Column "FLOORSMIN_MEDI" has 66.61% missing
Column "FLOORSMIN_AVG" has 66.61% missing
Column "FLOORSMIN_MODE" has 66.61% missing
Column "OWN_CAR_AGE" has 66.29% missing
Column "YEARS_BUILD_MODE" has 65.28% missing
Column "YEARS_BUILD_AVG" has 65.28% missing
Column "YEARS_BUILD_MEDI" has 65.28% missing
Column "LANDAREA_MEDI" has 57.96% missing
Column "LANDAREA_MODE" has 57.96% missing
Column "LANDAREA_AVG" has 57.96% missing
Column "BASEMENTAREA_AVG" has 56.71% missing
Column "BASEMENTAREA_MODE" has 56.71% missi

In [6]:
app_train.drop(columns=["APARTMENTS_MEDI","APARTMENTS_MODE"], inplace=True)
app_test.drop(columns=["APARTMENTS_MEDI","APARTMENTS_MODE"], inplace=True)

In [7]:
app_train["TARGET"].value_counts()

TARGET
0    282686
1     24825
Name: count, dtype: int64

In [8]:
app_train.dtypes.value_counts()

float64    63
int64      41
object     16
Name: count, dtype: int64

In [9]:
app_train['FLAG_OWN_CAR'] = app_train['FLAG_OWN_CAR'].apply(lambda x: 1 if x == 'Y' else 0)
app_test['FLAG_OWN_CAR'] = app_test['FLAG_OWN_CAR'].apply(lambda x: 1 if x == 'Y' else 0)
app_train['FLAG_OWN_REALTY'] = app_train['FLAG_OWN_REALTY'].apply(lambda x: 1 if x == 'Y' else 0)
app_test['FLAG_OWN_REALTY'] = app_test['FLAG_OWN_REALTY'].apply(lambda x: 1 if x == 'Y' else 0)

In [10]:
float64_columns = app_train.select_dtypes(include=['float64']).columns
app_train[float64_columns] = app_train[float64_columns].astype('float32')

int64_columns = app_train.select_dtypes(include=['int64']).columns
app_train[int64_columns] = app_train[int64_columns].astype('int32')

float64_columns = app_test.select_dtypes(include=['float64']).columns
app_test[float64_columns] = app_test[float64_columns].astype('float32')

int64_columns = app_test.select_dtypes(include=['int64']).columns
app_test[int64_columns] = app_test[int64_columns].astype('int32')

In [11]:
app_train.select_dtypes('object').apply(pd.Series.nunique, axis = 0)

NAME_CONTRACT_TYPE             2
CODE_GENDER                    3
NAME_TYPE_SUITE                7
NAME_INCOME_TYPE               8
NAME_EDUCATION_TYPE            5
NAME_FAMILY_STATUS             6
NAME_HOUSING_TYPE              6
OCCUPATION_TYPE               18
WEEKDAY_APPR_PROCESS_START     7
ORGANIZATION_TYPE             58
FONDKAPREMONT_MODE             4
HOUSETYPE_MODE                 3
WALLSMATERIAL_MODE             7
EMERGENCYSTATE_MODE            2
dtype: int64

In [12]:
app_train.query('OWN_CAR_AGE > 60')['OWN_CAR_AGE'].value_counts()

OWN_CAR_AGE
64.0    2443
65.0     891
63.0       2
91.0       2
69.0       1
Name: count, dtype: int64

In [13]:
app_train['OWN_CAR_AGE'] = app_train['OWN_CAR_AGE'].apply(lambda x: np.nan if x == 64 or x == 65 else x)
app_test['OWN_CAR_AGE'] = app_test['OWN_CAR_AGE'].apply(lambda x: np.nan if x == 64 or x == 65 else x)

In [14]:
app_train['AGE'] = app_train['DAYS_BIRTH']/-365
app_test['AGE'] = app_test['DAYS_BIRTH']/-365

C:\Users\myself\AppData\Local\Temp\ipykernel_19272\2471324857.py:1: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\myself\AppData\Local\Temp\ipykernel_19272\2471324857.py:2: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



In [15]:
(app_train['DAYS_EMPLOYED'].abs() > app_train['DAYS_BIRTH'].abs()).sum()

np.int64(55374)

In [16]:
app_train = app_train.copy()
app_test = app_test.copy()

In [17]:
#anamoly flag
app_train['DAYS_EMPLOYED_ANOM'] = app_train['DAYS_EMPLOYED'] == 365243
app_test['DAYS_EMPLOYED_ANOM'] = app_test['DAYS_EMPLOYED'] == 365243

#handling anamoly
app_train['DAYS_EMPLOYED'] = app_train['DAYS_EMPLOYED'].replace(365243, np.nan)
app_test['DAYS_EMPLOYED'] = app_test['DAYS_EMPLOYED'].replace(365243, np.nan)

In [18]:
app_train = app_train.copy()
app_test = app_test.copy()

In [19]:
app_train['CODE_GENDER'].value_counts()

CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64

In [20]:
app_test['CODE_GENDER'].value_counts()

CODE_GENDER
F    32678
M    16066
Name: count, dtype: int64

In [21]:
app_train['CODE_GENDER'].unique()

array(['M', 'F', 'XNA'], dtype=object)

In [22]:
app_train['CREDIT_INCOME_RATIO'] = app_train['AMT_CREDIT'] / app_train['AMT_INCOME_TOTAL']
app_test['CREDIT_INCOME_RATIO'] = app_test['AMT_CREDIT'] / app_test['AMT_INCOME_TOTAL']

app_train['ANNUITY_INCOME_RATIO'] = app_train['AMT_ANNUITY'] / app_train['AMT_INCOME_TOTAL']
app_test['ANNUITY_INCOME_RATIO'] = app_test['AMT_ANNUITY'] / app_test['AMT_INCOME_TOTAL']

In [23]:
app_train['CREDIT_ANNUITY_RATIO'] = app_train['AMT_CREDIT'] / app_train['AMT_ANNUITY']
app_test['CREDIT_ANNUITY_RATIO'] = app_test['AMT_CREDIT'] / app_test['AMT_ANNUITY']

In [24]:
app_train['INCOME_PER_PERSON'] = app_train['AMT_INCOME_TOTAL'] / app_train['CNT_FAM_MEMBERS']
app_test['INCOME_PER_PERSON'] = app_test['AMT_INCOME_TOTAL'] / app_test['CNT_FAM_MEMBERS']

In [25]:
ext_cols = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']

app_train['EXT_SOURCE_MEAN'] = app_train[ext_cols].mean(axis=1)
app_test['EXT_SOURCE_MEAN'] = app_test[ext_cols].mean(axis=1)

In [26]:
bureau_counts = bureau.groupby("SK_ID_CURR").size()

bureau_counts = bureau_counts.rename("BUREAU_LOAN_COUNT")

In [27]:
bureau_counts

SK_ID_CURR
100001     7
100002     8
100003     4
100004     2
100005     3
          ..
456249    13
456250     3
456253     4
456254     1
456255    11
Name: BUREAU_LOAN_COUNT, Length: 305811, dtype: int64

In [28]:
bureau_agg = bureau.groupby("SK_ID_CURR").agg({
    
    "DAYS_CREDIT": ["mean","min","max"],
    
    "AMT_CREDIT_SUM": ["sum","mean"],
    
    "AMT_CREDIT_SUM_DEBT": ["sum","mean"],
    
    "AMT_CREDIT_SUM_OVERDUE": ["sum"],
    
})

In [29]:
bureau_agg["BUREAU_LOAN_COUNT"] = bureau_counts

In [30]:
bureau_agg.columns = [
    "_".join(col).upper() for col in bureau_agg.columns
]

In [31]:
bureau['CREDIT_ACTIVE'].value_counts()

CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64

In [32]:
active_loans = bureau[bureau['CREDIT_ACTIVE'] == 'Active'] \
                .groupby('SK_ID_CURR') \
                .size() \
                .rename('ACTIVE_LOAN_COUNT')

In [33]:
app_train = app_train.merge(
    bureau_agg,
    how="left",
    on="SK_ID_CURR"
)

app_test = app_test.merge(
    bureau_agg,
    how="left",
    on="SK_ID_CURR"
)

app_train = app_train.merge(active_loans, how='left', on='SK_ID_CURR')
app_test  = app_test.merge(active_loans, how='left', on='SK_ID_CURR')

In [34]:
#app_train['BUREAU_LOAN_COUNT_'] = app_train['BUREAU_LOAN_COUNT_'].fillna(0)
#app_test['BUREAU_LOAN_COUNT_'] = app_test['BUREAU_LOAN_COUNT_'].fillna(0)

#app_train['ACTIVE_LOAN_COUNT'] = app_train['ACTIVE_LOAN_COUNT'].fillna(0)
#app_test['ACTIVE_LOAN_COUNT'] = app_test['ACTIVE_LOAN_COUNT'].fillna(0)

In [35]:
bureau_bal.head()

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [36]:
bureau_bal['STATUS'].value_counts()

STATUS
C    13646993
0     7499507
X     5810482
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64

In [37]:
bureau_bal['STATUS_NUM'] = bureau_bal['STATUS'].replace({
    'X': np.nan,
    'C': 0,
    '0': 0,
    '1': 1,
    '2': 2,
    '3': 3,
    '4': 4,
    '5': 5
})

C:\Users\myself\AppData\Local\Temp\ipykernel_19272\1406404971.py:1: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [38]:
bureau_bal_agg = bureau_bal.groupby('SK_ID_BUREAU').agg({
    
    'MONTHS_BALANCE': ['min','max'],
    
    'STATUS_NUM': ['mean','max']
    
})

In [39]:
bureau_bal_agg.columns = [
    "_".join(col).upper() for col in bureau_bal_agg.columns
]

In [40]:
bureau = bureau.merge(
    bureau_bal_agg,
    how='left',
    on='SK_ID_BUREAU'
)

In [41]:
bureau_balance_features = bureau.groupby('SK_ID_CURR').agg({
    
    'STATUS_NUM_MAX': ['max','mean'],
    
    'MONTHS_BALANCE_MIN': ['min']
    
})

In [42]:
bureau_balance_features.columns = [
    "_".join(col).upper() for col in bureau_balance_features.columns
]

In [43]:
app_train = app_train.merge(
    bureau_balance_features,
    how='left',
    on='SK_ID_CURR'
)

app_test = app_test.merge(
    bureau_balance_features,
    how='left',
    on='SK_ID_CURR'
)

In [44]:
#app_train[bureau_balance_features.columns] = app_train[bureau_balance_features.columns].fillna(0)
#app_test[bureau_balance_features.columns] = app_test[bureau_balance_features.columns].fillna(0)

In [45]:
#del bureau
#del bureau_agg
#del bureau_counts

In [46]:
prev_app.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
prev_count = prev_app.groupby("SK_ID_CURR").size().rename("PREV_APP_COUNT")

In [48]:
approved = prev_app[prev_app['NAME_CONTRACT_STATUS']=='Approved'] \
           .groupby('SK_ID_CURR').size().rename('APPROVED_COUNT')

refused = prev_app[prev_app['NAME_CONTRACT_STATUS']=='Refused'] \
          .groupby('SK_ID_CURR').size().rename('REFUSED_COUNT')

In [49]:
prev_agg = prev_app.groupby("SK_ID_CURR").agg({

    "AMT_APPLICATION": ["mean","max"],
    "AMT_CREDIT": ["mean","max"],
    "AMT_ANNUITY": ["mean"],
    "DAYS_DECISION": ["mean","min"]

})

In [50]:
prev_agg.columns = [
    "_".join(col).upper() for col in prev_agg.columns
]

In [51]:
app_train = app_train.merge(prev_agg, how='left', on='SK_ID_CURR')
app_test  = app_test.merge(prev_agg, how='left', on='SK_ID_CURR')

app_train = app_train.merge(prev_count, how='left', on='SK_ID_CURR')
app_test  = app_test.merge(prev_count, how='left', on='SK_ID_CURR')

app_train = app_train.merge(approved, how='left', on='SK_ID_CURR')
app_test  = app_test.merge(approved, how='left', on='SK_ID_CURR')

app_train = app_train.merge(refused, how='left', on='SK_ID_CURR')
app_test  = app_test.merge(refused, how='left', on='SK_ID_CURR')

In [52]:
cols = ['PREV_APP_COUNT','APPROVED_COUNT','REFUSED_COUNT']

#app_train[cols] = app_train[cols].fillna(0)
#app_test[cols] = app_test[cols].fillna(0)

In [53]:
pos_agg = pos.groupby("SK_ID_CURR").agg({

    "MONTHS_BALANCE": ["min","max"],
    "CNT_INSTALMENT": ["mean","max"],
    "CNT_INSTALMENT_FUTURE": ["mean"],
    "SK_DPD": ["max","mean"],
    "SK_DPD_DEF": ["max"]

})

In [54]:
pos_agg.columns = ["_".join(col).upper() for col in pos_agg.columns]

In [55]:
app_train = app_train.merge(pos_agg, how="left", on="SK_ID_CURR")
app_test  = app_test.merge(pos_agg, how="left", on="SK_ID_CURR")

In [56]:
installments["PAYMENT_DELAY"] = installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"]

In [57]:
inst_agg = installments.groupby("SK_ID_CURR").agg({

    "PAYMENT_DELAY": ["mean","max"],
    "AMT_PAYMENT": ["sum","mean"],
    "AMT_INSTALMENT": ["sum","mean"]

})

In [58]:
inst_agg.columns = ["_".join(col).upper() for col in inst_agg.columns]

In [59]:
app_train = app_train.merge(inst_agg, how="left", on="SK_ID_CURR")
app_test  = app_test.merge(inst_agg, how="left", on="SK_ID_CURR")

In [60]:
cc_agg = credit_card.groupby("SK_ID_CURR").agg({

    "AMT_BALANCE": ["mean","max"],
    "AMT_CREDIT_LIMIT_ACTUAL": ["mean"],
    "SK_DPD": ["max","mean"]

})

In [61]:
cc_agg.columns = ["_".join(col).upper() for col in cc_agg.columns]

In [62]:
app_train = app_train.merge(cc_agg, how="left", on="SK_ID_CURR")
app_test  = app_test.merge(cc_agg, how="left", on="SK_ID_CURR")

In [63]:
print("Train shape:", app_train.shape)
print("Test shape:", app_test.shape)

Train shape: (307511, 169)
Test shape: (48744, 168)


In [64]:
print("Duplicate columns:", app_train.columns.duplicated().sum())

Duplicate columns: 0


In [65]:
constant_cols = [col for col in app_train.columns if app_train[col].nunique() <= 1]

print("Constant columns:", len(constant_cols))

Constant columns: 0


In [66]:
app_train["TARGET"].value_counts(normalize=True)

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

In [67]:
y = app_train["TARGET"]

X = app_train.drop(columns=["TARGET"])
X_test = app_test.copy()

In [68]:
for col in X.select_dtypes("object").columns:
    print(col, X[col].nunique())

NAME_CONTRACT_TYPE 2
CODE_GENDER 3
NAME_TYPE_SUITE 7
NAME_INCOME_TYPE 8
NAME_EDUCATION_TYPE 5
NAME_FAMILY_STATUS 6
NAME_HOUSING_TYPE 6
OCCUPATION_TYPE 18
WEEKDAY_APPR_PROCESS_START 7
ORGANIZATION_TYPE 58
FONDKAPREMONT_MODE 4
HOUSETYPE_MODE 3
WALLSMATERIAL_MODE 7
EMERGENCYSTATE_MODE 2


In [69]:
def encoding_groups(df, target_col="TARGET", ohe_threshold=10, label_threshold=50):
    
    ohe_cols = []
    label_cols = []
    target_cols = []
    
    cat_cols = df.select_dtypes(include=["object"]).columns
    
    for col in cat_cols:
        
        n_unique = df[col].nunique()
        
        if n_unique <= ohe_threshold:
            ohe_cols.append(col)
            
        elif n_unique <= label_threshold:
            label_cols.append(col)
            
        else:
            target_cols.append(col)
    
    return ohe_cols, label_cols, target_cols

In [70]:
ohe_cols, label_cols, target_cols = encoding_groups(app_train)
label_cols = label_cols + target_cols
print("OHE columns:", len(ohe_cols))
print("Label columns:", len(label_cols))
print("Target columns:", len(target_cols))

OHE columns: 12
Label columns: 2
Target columns: 1


In [71]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# detect column types
num_cols = X.select_dtypes(exclude="object").columns
cat_cols = X.select_dtypes(include="object").columns

# numeric pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# categorical pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# final preprocessor
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

In [72]:
X = X.drop(columns=["SK_ID_CURR"])
X_test = X_test.drop(columns=["SK_ID_CURR"])

In [73]:
from sklearn.pipeline import Pipeline
import lightgbm as lgb

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", lgb.LGBMClassifier(
    n_estimators=5000,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=64,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
))
    ]
)

In [74]:
from sklearn.model_selection import StratifiedKFold

In [75]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [76]:
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\nFold {fold+1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(
        X_train,
        y_train,
        model__eval_set=[(X_val, y_val)],
        model__eval_metric="auc",
        model__callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(200)
        ]
    )

    val_pred = model.predict_proba(X_val)[:,1]

    oof_preds[val_idx] = val_pred

    test_preds += model.predict_proba(X_test)[:,1] / skf.n_splits


Fold 1


ValueError: A given column is not a column of the dataframe

In [ ]:
print("CV ROC-AUC:", roc_auc_score(y, oof_preds))

In [ ]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()

In [ ]:
importances = model.named_steps["model"].feature_importances_

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

In [ ]:
feat_imp = feat_imp.sort_values("importance", ascending=False)

print(feat_imp)

In [ ]:
[id_col for id_col in X.columns if "SK_ID" in id_col]

In [ ]:
import xgboost as xgb

xgb_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", xgb.XGBClassifier(
            n_estimators=5000,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="auc",
            tree_method="hist",
            early_stopping_rounds=100,
            random_state=42
        ))
    ]
)

In [ ]:
oof_preds_xgb = np.zeros(len(X))
test_preds_xgb = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\nXGBoost Fold {fold+1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    xgb_model.named_steps["preprocess"].fit(X_train)

    X_train_p = xgb_model.named_steps["preprocess"].transform(X_train)
    X_val_p = xgb_model.named_steps["preprocess"].transform(X_val)
    X_test_p = xgb_model.named_steps["preprocess"].transform(X_test)

    xgb_model.named_steps["model"].fit(
        X_train_p,
        y_train,
        eval_set=[(X_val_p, y_val)],
        verbose=200
    )

    val_pred = xgb_model.named_steps["model"].predict_proba(X_val_p)[:,1]
    oof_preds_xgb[val_idx] = val_pred

    test_preds_xgb += (
        xgb_model.named_steps["model"].predict_proba(X_test_p)[:,1] 
        / skf.n_splits
    )

In [ ]:
print("XGBoost CV ROC-AUC:", roc_auc_score(y, oof_preds_xgb))

In [ ]:
blend_preds = 0.5 * oof_preds + 0.5 * oof_preds_xgb

print("Blended ROC-AUC:", roc_auc_score(y, blend_preds))

In [ ]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.05,
    depth=6,
    eval_metric="AUC",
    random_state=42
)

In [ ]:
oof_cat = np.zeros(len(X))
test_cat = np.zeros(len(X_test))

fold_scores = []

for train_idx, val_idx in skf.split(X, y):

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    cat_model.fit(
        X_train,
        y_train,
        cat_features=[X.columns.get_loc(col) for col in cat_cols],
        eval_set = (X_val, y_val),
        early_stopping_rounds = 100,
        verbose = 200
    )

    val_pred = cat_model.predict_proba(X_val)[:,1]

    score = roc_auc_score(y_val, val_pred)
    fold_scores.append(score)

    oof_cat[val_idx] = val_pred
    test_cat += cat_model.predict_proba(X_test)[:,1] / skf.n_splits

In [ ]:
print("CatBoost fold scores:", fold_scores)
print("CatBoost CV:", np.mean(fold_scores))

In [ ]:
print("CatBoost CV ROC-AUC:", roc_auc_score(y, oof_cat))

In [ ]:
from sklearn.metrics import average_precision_score

print("LightGBM ROC-AUC:", roc_auc_score(y, oof_preds))
print("LightGBM PR-AUC:", average_precision_score(y, oof_preds))

In [ ]:
print("XGBoost ROC-AUC:", roc_auc_score(y, oof_preds_xgb))
print("XGBoost PR-AUC:", average_precision_score(y, oof_preds_xgb))

In [ ]:
print("CatBoost ROC-AUC:", roc_auc_score(y, oof_cat))
print("CatBoost PR-AUC:", average_precision_score(y, oof_cat))

In [ ]:
blend = 0.4 * oof_preds + 0.4 * oof_preds_xgb + 0.2 * oof_cat

print("Blend ROC-AUC:", roc_auc_score(y, blend))
print("Blend PR-AUC:", average_precision_score(y, blend))

In [ ]:
# split categorical and numeric
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# encode categorical
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_cat = encoder.fit_transform(X[cat_cols])
X_test_cat = encoder.transform(X_test[cat_cols])

# numeric imputation
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_num = imputer.fit_transform(X[num_cols])
X_test_num = imputer.transform(X_test[num_cols])

# combine
X_nn = np.hstack([X_num, X_cat])
X_test_nn = np.hstack([X_test_num, X_test_cat])

# replace any remaining NaNs
X_nn = np.nan_to_num(X_nn)
X_test_nn = np.nan_to_num(X_test_nn)

# scale features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_nn = scaler.fit_transform(X_nn)
X_test_nn = scaler.transform(X_test_nn)

In [ ]:
import torch

X_nn = torch.tensor(X_nn, dtype=torch.float32)
X_test_nn = torch.tensor(X_test_nn, dtype=torch.float32)
y_nn = torch.tensor(y.values, dtype=torch.float32)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
import torch.nn as nn

class CreditNN(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.model = nn.Sequential(

            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
oof_nn = np.zeros(len(X))
test_nn = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\nNN Fold {fold+1}")

    X_train = X_nn[train_idx].to(device)
    X_val = X_nn[val_idx].to(device)

    y_train = y_nn[train_idx].unsqueeze(1).to(device)

    nn_model = CreditNN(X_nn.shape[1]).to(device)

    optimizer = torch.optim.Adam(nn_model.parameters(), lr=3e-4)
    criterion = nn.BCEWithLogitsLoss()

    epochs = 10
    batch_size = 1024

    for epoch in range(epochs):

        nn_model.train()

        perm = torch.randperm(X_train.size(0))

        for i in range(0, X_train.size(0), batch_size):

            idx = perm[i:i+batch_size]

            xb = X_train[idx]
            yb = y_train[idx]

            optimizer.zero_grad()

            preds = nn_model(xb)

            loss = criterion(preds, yb)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(nn_model.parameters(), 1.0)

            optimizer.step()

        print(f"Epoch {epoch+1} Loss {loss.item():.4f}")

    nn_model.eval()

    with torch.no_grad():

        val_preds = torch.sigmoid(nn_model(X_val)).cpu().numpy().flatten()

        oof_nn[val_idx] = val_preds

        test_preds = torch.sigmoid(nn_model(X_test_nn.to(device))).cpu().numpy().flatten()

        test_nn += test_preds / skf.n_splits

In [ ]:
print("NN ROC-AUC:", roc_auc_score(y, oof_nn))

In [ ]:
import pandas as pd

feature_names = preprocessor.get_feature_names_out()

lgb_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": lgb_model.feature_importances_
}).sort_values("importance", ascending=False)

print(lgb_importance.head(20))

In [ ]:
xgb_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_model.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

display(xgb_importance.head(20))

In [ ]:
cat_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": cat_model.get_feature_importance()
}).sort_values("importance", ascending=False)

print(cat_importance.head(20))

In [ ]:
weights = nn_model.model[0].weight.detach().cpu().numpy()

nn_importance = pd.DataFrame({
    "feature": list(num_cols) + list(cat_cols),
    "importance": abs(weights).mean(axis=0)
}).sort_values("importance", ascending=False)

print(nn_importance.head(20))

In [ ]:
import seaborn as sns

sns.barplot(
    data=lgb_importance.head(20),
    x="importance",
    y="feature"
)

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


def evaluate_model(y_true, preds, name="Model", threshold=0.5):

    labels = (preds >= threshold).astype(int)

    print(f"\n{name} Performance")
    print("-"*30)

    print("ROC-AUC:", roc_auc_score(y_true, preds))
    print("PR-AUC:", average_precision_score(y_true, preds))
    print("LogLoss:", log_loss(y_true, preds))
    print("Accuracy:", accuracy_score(y_true, labels))
    print("Precision:", precision_score(y_true, labels))
    print("Recall:", recall_score(y_true, labels))
    print("F1:", f1_score(y_true, labels))

    # ROC curve
    fpr, tpr, _ = roc_curve(y_true, preds)

    plt.figure(figsize=(6,4))
    plt.plot(fpr, tpr, label=name)
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{name} ROC Curve")
    plt.legend()
    plt.show()

    # PR curve
    precision, recall, _ = precision_recall_curve(y_true, preds)

    plt.figure(figsize=(6,4))
    plt.plot(recall, precision, label=name)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{name} Precision-Recall Curve")
    plt.legend()
    plt.show()

    # Confusion matrix
    cm = confusion_matrix(y_true, labels)

    plt.figure(figsize=(4,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
print("\nLightGBM Performance")
evaluate_model(y, oof_preds)

In [ ]:
print("\nXGBoost Performance")
evaluate_model(y, oof_preds_xgb)

In [ ]:
print("\nCatBoost Performance")
evaluate_model(y, oof_cat)

In [ ]:
print("\nNeural Network Performance")
evaluate_model(y, oof_nn)

In [ ]:
blend = 0.4 * oof_preds + 0.4 * oof_preds_xgb + 0.2 * oof_cat

print("\nBlended Model Performance")
evaluate_model(y, blend)

In [ ]:
print("Final Model Summary")

print("LightGBM ROC-AUC:", roc_auc_score(y, oof_preds))
print("XGBoost ROC-AUC:", roc_auc_score(y, oof_preds_xgb))
print("CatBoost ROC-AUC:", roc_auc_score(y, oof_cat))
print("Neural Network ROC-AUC:", roc_auc_score(y, oof_nn))

blend = 0.4 * oof_preds + 0.4 * oof_preds_xgb + 0.2 * oof_cat

print("\nFinal Blended ROC-AUC:", roc_auc_score(y, blend))

In [ ]:
import shap

explainer = shap.TreeExplainer(lgb_model)

X_val_df = pd.DataFrame(
    X_val_processed,
    columns=feature_names
).astype(float)

shap_values = explainer.shap_values(X_val_df)

shap.summary_plot(shap_values, X_val_df)

In [ ]:
stack_train = pd.DataFrame({
    "lgb": oof_preds,
    "xgb": oof_preds_xgb,
    "cat": oof_cat
})

stack_train.head()

In [ ]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression()

meta_model.fit(stack_train, y)

In [ ]:
stack_preds = meta_model.predict_proba(stack_train)[:,1]

print("Stacked ROC-AUC:", roc_auc_score(y, stack_preds))
print("Stacked PR-AUC:", average_precision_score(y, stack_preds))

In [ ]:
stack_test = pd.DataFrame({
    "lgb": test_preds,
    "xgb": test_preds_xgb,
    "cat": test_cat
})

In [ ]:
final_test_preds = meta_model.predict_proba(stack_test)[:,1]

In [ ]:
import joblib

joblib.dump(meta_model, "stacking_model.pkl")
joblib.dump(lgb_model, "lgb_model.pkl")
joblib.dump(xgb_model.named_steps["model"], "xgb_model.pkl")
joblib.dump(cat_model, "cat_model.pkl")

print("All models saved.")

In [3]:
import joblib
cat_model = joblib.load("models/cat_model.pkl")
print(type(cat_model.feature_names_))
print(cat_model.feature_names_[:10])
print(cat_model.get_cat_feature_indices()[:10])

FileNotFoundError: [Errno 2] No such file or directory: 'models/cat_model.pkl'